# Cycle topology — RQ2 campaignRing network (0-1-2-3-0): every agent has exactly **2** neighbours, so it isdegree-regular like the clique but sparse like the line. That separates*symmetry* from *connectivity* — the star cannot.**Before running:** Accelerator = `GPU T4 x2`, Internet = `On`, and a KaggleSecret named `HF_TOKEN` for gated models (Gemma, Llama).Pick `MODEL_ID` and `SESSION` in the config cell; everything else is derived.Each model needs two sessions (A then B) except Qwen3-4B, which fits in one.For runs over ~2 h use **Save Version -> Save & Run All**, so a browserdisconnect cannot kill the session.

In [ ]:
!pip install -q bitsandbytes python-dotenvimport torch, transformersassert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4"print(f"GPU:   {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")print(f"torch: {torch.__version__}   transformers: {transformers.__version__}")

In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"import os, subprocessif os.path.exists('/kaggle/working/repo'):    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "--ff-only"], check=True)else:    subprocess.run(["git", "clone", GITHUB_REPO, "/kaggle/working/repo"], check=True)%cd /kaggle/working/repoprint("HEAD:", subprocess.run(["git","log","--oneline","-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
import ostry:    from kaggle_secrets import UserSecretsClient    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')    print('HF token loaded (needed for Gemma / Llama)')except Exception as e:    print(f'No HF_TOKEN secret (fine for Qwen): {e}')

## Config — the only cell you edit`MAX_NEW_TOKENS` is matched to each model's **star** setting so thestar-vs-cycle contrast carries no extra confound. Do not "tidy" these numbers.

In [ ]:
MODEL_ID = "google/gemma-2-2b-it"# MODEL_ID = "Qwen/Qwen3-4B"              # the only model that fits A+B in one session# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"# MODEL_ID = "google/gemma-2-9b-it"SESSION  = "A"      # "A" = baseline+no_sense+silence+counterfactual, "B" = the 3 framingsTOPOLOGY = "cycle"N_RUNS, N_ROUNDS = 5, 16      # protocol rule: 5 runs everywhereSCENARIOS = {    "A": ["baseline", "no_sense", "silence", "counterfactual"],    "B": ["framing_business", "framing_team", "framing_competitive"],}[SESSION]# max_new_tokens per model, per session, copied from the star campaign.MAX_NEW_TOKENS = {    "meta-llama/Llama-3.1-8B-Instruct": {"A": 192, "B": 192},    "Qwen/Qwen2.5-7B-Instruct":         {"A": 512, "B": 256},    "Qwen/Qwen3-4B":                    {"A": 256, "B": 256},    "google/gemma-2-2b-it":             {"A": 256, "B": 192},    "google/gemma-2-9b-it":             {"A": 160, "B": 192},}[MODEL_ID][SESSION]MODEL_SHORT = MODEL_ID.split("/")[-1]OUT_DIR_BASE = f"/kaggle/working/results/{MODEL_SHORT}"# run_all_scenarios.py appends the topology suffix itself for non-star runs:RESULT_DIR = OUT_DIR_BASE if TOPOLOGY == "star" else f"{OUT_DIR_BASE}_{TOPOLOGY}"print(f"Model      : {MODEL_ID}")print(f"Session {SESSION}  : {SCENARIOS}")print(f"Topology   : {TOPOLOGY}   runs={N_RUNS} rounds={N_ROUNDS} max_new_tokens={MAX_NEW_TOKENS}")print(f"Results -> : {RESULT_DIR}")

## RunBuilt as an argument list and executed with `subprocess`, deliberately **not**`!python ... $VAR`: shell magic silently expands an undefined/misspelledvariable to an empty string, which is how a run dies with`error: argument --model-id: expected one argument` while every print above itlooks correct. This cell raises instead of letting the notebook continue to thezip step with no data.

In [ ]:
import subprocess, sys, shlex, timecmd = [    sys.executable, "run_all_scenarios.py",    "--provider", "local",    "--model-id", MODEL_ID,    "--topology", TOPOLOGY,    "--n-runs", str(N_RUNS),    "--n-rounds", str(N_ROUNDS),    "--max-new-tokens", str(MAX_NEW_TOKENS),    "--out-dir-base", OUT_DIR_BASE,    "--zip-mirror", "/kaggle/working",    "--no-probe",    "--scenarios", *SCENARIOS,]print("RUN:", " ".join(shlex.quote(c) for c in cmd), flush=True)t0 = time.time()rc = subprocess.run(cmd).returncodeelapsed = time.time() - t0print(f"exit code {rc} after {elapsed/60:.1f} min ({elapsed/3600:.2f} h)")if rc != 0:    raise SystemExit(f"run_all_scenarios.py FAILED (exit {rc}) — stopping, nothing to package.")

## Verify, then packageCounts the JSON files actually produced and compares against what the protocoldemands, so a half-finished session is visible here rather than three weekslater in the analysis.

In [ ]:
import glob, json, os, shutilfrom collections import Counterfiles = glob.glob(os.path.join(RESULT_DIR, "**", "*.json"), recursive=True)files = [f for f in files if not os.path.basename(f).startswith("_progress")]expected = len(SCENARIOS) * 2 * N_RUNS + (2 * N_RUNS if "baseline" in SCENARIOS else 0)per_scenario = Counter()topologies = Counter()for f in files:    rec = json.load(open(f))    per_scenario[os.path.relpath(f, RESULT_DIR).split(os.sep)[0]] += 1    topologies[rec["topology"]["type"]] += 1print(f"JSON runs found : {len(files)}  (expected {expected})")print(f"topologies      : {dict(topologies)}")for k, v in sorted(per_scenario.items()):    print(f"   {k:24s} {v:3d} runs")assert files, "No runs were produced."assert set(topologies) == {TOPOLOGY}, f"Wrong topology in records: {dict(topologies)}"if len(files) != expected:    print(f"WARNING: expected {expected} runs, got {len(files)} — a scenario may be incomplete.")zip_path = f"/kaggle/working/{MODEL_SHORT}_{TOPOLOGY}_session{SESSION}"shutil.make_archive(zip_path, "zip", RESULT_DIR)print(f"Download this: {zip_path}.zip  ({os.path.getsize(zip_path + '.zip')/1e6:.1f} MB)")

## After downloading1. Put the zip in `diplomatikh/drive_sync/` and upload it to the Drive folder   `cheaptalk_bench_results`.2. Append the runs to the `Runs` sheet of `cheaptalk_results_tracker.xlsx` —   `Combinations` and `Results` recompute themselves.3. Add a row to `TRACK_RECORD.md`, noting the wall-clock time from the run cell   so the remaining estimates can be recalibrated.